# 01 - Fundamentos de BigQuery

**Fase 1 - NYC Taxi Data Science Project**

---

## Pregunta de negocio

> **¿Cuánto cuesta consultar 1.5 mil millones de viajes?**

Antes de lanzarnos a construir modelos, necesitamos entender nuestro almacén de datos:
cuánto mide, cuánto cuesta consultarlo y cómo obtener muestras representativas sin
arruinarnos.

### Dataset

| Propiedad | Valor |
|-----------|-------|
| Tabla | `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015` |
| Filas aprox. | ~146 millones |
| Año | 2015 |
| Contenido | Viajes de taxis amarillos en NYC con zonas TLC |

### Lo que aprenderemos

1. Configuración y autenticación con Google Cloud
2. Conexión a BigQuery usando nuestro helper
3. Exploración del esquema del dataset
4. Estrategias de muestreo (TABLESAMPLE, RAND, FARM_FINGERPRINT, LIMIT)
5. Estimación de costos con dry_run
6. Operaciones básicas con NumPy/Pandas sobre resultados
7. Mejores prácticas para trabajar con datos masivos

---

## 1. Setup y Prerrequisitos

### 1.1 BigQuery Sandbox (Tier Gratuito)

Google Cloud ofrece un **sandbox de BigQuery** que no requiere tarjeta de crédito:

| Recurso | Límite gratuito mensual |
|---------|------------------------|
| Consultas (procesamiento) | **1 TB/mes** |
| Almacenamiento | **10 GB** |
| Datos públicos | Sin costo adicional |

**Importante:** El sandbox es suficiente para este proyecto completo. Con 1 TB/mes de
consultas gratuitas y las técnicas de muestreo que veremos aquí, nunca deberías pagar
un centavo.

### 1.2 Instalación de dependencias

Ejecuta las siguientes celdas solo si aún no tienes los paquetes instalados.

In [1]:
# Instalar dependencias (ejecutar solo la primera vez)
# !pip install google-cloud-bigquery db-dtypes pyarrow pandas numpy

### 1.3 Autenticación con Google Cloud

Necesitas autenticarte con `gcloud` antes de usar BigQuery. Ejecuta esto **una sola vez**
en tu terminal (no en el notebook):

```bash
# 1. Instalar gcloud CLI (si no lo tienes):
#    https://cloud.google.com/sdk/docs/install

# 2. Iniciar sesión:
gcloud auth login

# 3. Configurar credenciales por defecto para aplicaciones:
gcloud auth application-default login

# 4. (Opcional) Establecer un proyecto por defecto:
gcloud config set project TU_PROYECTO_ID
```

Si usas el sandbox sin proyecto propio, puedes pasar `project_id` directamente
al crear el helper.

---

## 2. Conexión a BigQuery

Usamos nuestro helper personalizado (`src/bigquery/bq_helper.py`) que nos da:

- **Cache local en Parquet** para no repetir consultas costosas
- **Estimación de costos** automática antes de ejecutar
- **Control de gasto** con límite configurable por query
- **Muestreo reproducible** con FARM_FINGERPRINT

In [2]:
import sys
sys.path.insert(0, '../../../src')

from bigquery.bq_helper import BigQueryHelper

bq = BigQueryHelper()

✓ Conectado a BigQuery - Proyecto: gen-lang-client-0180273702
✓ Cache en: /home/danielmf31/Documentos/Documentos_Trabajo/Ingenieria/Programacion/VSCode/Proyectos_personales/Ciencia_Datos/data/nyc_taxi/cache


Si todo salió bien, deberías ver un mensaje confirmando la conexión y el directorio de cache.

> **Nota:** Si recibes un error de autenticación, revisa la sección 1.3 de este notebook.

---

## 3. Exploración del Esquema

Antes de escribir una sola query, necesitamos entender la **estructura** de nuestros datos.
Esto es gratis en BigQuery -- consultar metadatos no consume tu cuota.

In [3]:
# Obtener información de la tabla
info = bq.get_table_info()

📋 Tabla: bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015
   Filas: 146,039,220
   Tamaño: 25.78 GB
   Columnas: 20


In [4]:
# Ver el número exacto de filas y tamaño
print(f"Filas totales: {info['num_rows']:,}")
print(f"Tamaño en disco: {info['size_gb']:.2f} GB")
print(f"Fecha de creación: {info['created']}")

Filas totales: 146,039,220
Tamaño en disco: 25.78 GB
Fecha de creación: 2022-09-14 04:15:52.039000+00:00


In [5]:
import pandas as pd

# Convertir el esquema a DataFrame para visualización
schema_df = pd.DataFrame(info['schema'])
schema_df

,name,type,mode,description
0,vendor_id,STRING,REQUIRED,A code indicating the LPEP provider that provi...
1,pickup_datetime,TIMESTAMP,NULLABLE,The date and time when the meter was engaged
2,dropoff_datetime,TIMESTAMP,NULLABLE,The date and time when the meter was disengaged
3,passenger_count,INTEGER,NULLABLE,The number of passengers in the vehicle. This ...
4,trip_distance,NUMERIC,NULLABLE,The elapsed trip distance in miles reported by...
5,rate_code,STRING,NULLABLE,The final rate code in effect at the end of th...
6,store_and_fwd_flag,STRING,NULLABLE,This flag indicates whether the trip record wa...
7,payment_type,STRING,NULLABLE,A numeric code signifying how the passenger pa...
8,fare_amount,NUMERIC,NULLABLE,The time-and-distance fare calculated by the m...
9,extra,NUMERIC,NULLABLE,Miscellaneous extras and surcharges. Currently...


### Observaciones sobre el esquema

Puntos clave a notar:

- **Zonas geográficas:** `pickup_location_id`, `dropoff_location_id` (IDs de zonas TLC de NYC)
- **Temporales:** `pickup_datetime`, `dropoff_datetime` (timestamps completos)
- **Financieros:** `fare_amount`, `tip_amount`, `tolls_amount`, `total_amount`
- **Categóricos:** `vendor_id`, `rate_code`, `payment_type`
- **Métricos:** `passenger_count`, `trip_distance`

> **Nota:** Esta versión del dataset usa IDs de zonas TLC en lugar de coordenadas GPS.
> Cada `location_id` corresponde a una de las 263 zonas de taxi definidas por la TLC de NYC.

Con ~146M filas y este esquema, tenemos un dataset riquísimo para explorar patrones
geográficos, temporales y económicos.

---

## 4. Estimación de Costos

### ¿Cuánto cuesta consultar toda la tabla?

BigQuery cobra por **volumen de datos procesados**, no por tiempo de cómputo.
El precio on-demand es **$6.25 por TB** procesado.

Usemos `estimate_cost()` (internamente hace un `dry_run`) para ver cuánto costaría
un `SELECT *` sobre toda la tabla.

In [6]:
# Cuánto cuesta un SELECT * completo?
query_full = """
SELECT *
FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
"""

cost_full = bq.estimate_cost(query_full)
cost_full

📊 Estimación: 25.784 GB → $0.1574 USD


{'bytes_processed': 27684953580,
 'gb': 25.784,
 'tb': 0.025179,
 'estimated_cost_usd': 0.1574}

### Anatomía de un dry_run

El `dry_run` le pide a BigQuery que **analice** la query sin ejecutarla. Nos dice
exactamente cuántos bytes procesaría, lo cual es suficiente para calcular el costo.

```python
# Así funciona internamente nuestro helper:
job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
job = client.query(query, job_config=job_config)
bytes_procesados = job.total_bytes_processed
```

**Regla de oro:** Siempre ejecuta `estimate_cost()` antes de una query nueva.

In [7]:
# Cuánto cuesta si solo pedimos columnas específicas?
# BigQuery es columnar: solo cobra por las columnas que tocas
query_selective = """
SELECT
    pickup_datetime,
    dropoff_datetime,
    passenger_count,
    trip_distance,
    fare_amount,
    tip_amount,
    total_amount
FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
"""

cost_selective = bq.estimate_cost(query_selective)

# Comparar costos
print(f"\n--- Comparación ---")
print(f"SELECT * :         {cost_full['gb']:.2f} GB → ${cost_full['estimated_cost_usd']:.4f}")
print(f"SELECT 7 cols:     {cost_selective['gb']:.2f} GB → ${cost_selective['estimated_cost_usd']:.4f}")
print(f"Ahorro:            {(1 - cost_selective['gb']/cost_full['gb'])*100:.0f}%")

📊 Estimación: 11.969 GB → $0.0731 USD

--- Comparación ---
SELECT * :         25.78 GB → $0.1574
SELECT 7 cols:     11.97 GB → $0.0731
Ahorro:            54%


### Lección clave: BigQuery es columnar

A diferencia de una base de datos relacional tradicional (PostgreSQL, MySQL), BigQuery
almacena los datos **por columna**, no por fila. Esto significa que:

- `SELECT *` escanea TODAS las columnas (caro)
- `SELECT columna_1, columna_2` solo escanea esas 2 columnas (barato)
- Agregar un `WHERE` **no reduce** el costo si no hay particiones
- Agregar un `LIMIT` **no reduce** el costo de escaneo

> **Nunca uses `SELECT *` en producción.** Siempre selecciona solo las columnas que necesitas.

In [8]:
# Demostración: LIMIT no reduce el costo de escaneo
query_limit = """
SELECT *
FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
LIMIT 100
"""

cost_limit = bq.estimate_cost(query_limit)

print(f"\nSELECT * LIMIT 100: {cost_limit['gb']:.2f} GB → ${cost_limit['estimated_cost_usd']:.4f}")
print(f"SELECT * sin LIMIT: {cost_full['gb']:.2f} GB → ${cost_full['estimated_cost_usd']:.4f}")
print(f"\n¡LIMIT no ahorra nada! BigQuery escanea toda la tabla de todas formas.")

📊 Estimación: 25.784 GB → $0.1574 USD

SELECT * LIMIT 100: 25.78 GB → $0.1574
SELECT * sin LIMIT: 25.78 GB → $0.1574

¡LIMIT no ahorra nada! BigQuery escanea toda la tabla de todas formas.


---

## 5. Estrategias de Muestreo

Si LIMIT no reduce costos, ¿cómo obtenemos una muestra sin pagar por toda la tabla?

Veamos 4 estrategias, de la más simple a la más robusta.

### 5.1 TABLESAMPLE SYSTEM (la más barata)

`TABLESAMPLE SYSTEM` selecciona bloques enteros de datos al azar. Es la opción
**más eficiente en costos** porque BigQuery puede saltarse bloques completos sin leerlos.

```sql
SELECT * FROM tabla TABLESAMPLE SYSTEM (1 PERCENT)
```

**Ventajas:** Muy barato, rápido.  
**Desventajas:** No es un muestreo aleatorio verdadero (muestrea bloques), no es
reproducible entre ejecuciones, el porcentaje es aproximado.

In [9]:
# Costo de TABLESAMPLE vs tabla completa
query_tablesample = """
SELECT
    pickup_datetime,
    trip_distance,
    fare_amount,
    tip_amount,
    total_amount
FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
TABLESAMPLE SYSTEM (1 PERCENT)
"""

cost_ts = bq.estimate_cost(query_tablesample)
print(f"\nTABLESAMPLE 1%: ~{cost_ts['gb']:.2f} GB")
print("Nota: BigQuery reporta el tamaño completo en dry_run,")
print("pero en la ejecución real solo escanea el porcentaje indicado.")

📊 Estimación: 0.096 GB → $0.0006 USD

TABLESAMPLE 1%: ~0.10 GB
Nota: BigQuery reporta el tamaño completo en dry_run,
pero en la ejecución real solo escanea el porcentaje indicado.


### 5.2 RAND() (muestreo aleatorio verdadero)

Usa la función `RAND()` para generar un número aleatorio por fila y filtrar.

```sql
SELECT * FROM tabla WHERE RAND() < 0.01  -- ~1% de las filas
```

**Ventajas:** Muestreo uniforme verdadero.  
**Desventajas:** Escanea TODA la tabla (mismo costo que sin filtro). No reproducible
(cada ejecución da filas diferentes).

In [10]:
# RAND() escanea toda la tabla, así que el costo es igual
query_rand = """
SELECT
    pickup_datetime,
    trip_distance,
    fare_amount
FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
WHERE RAND() < 0.001  -- ~0.1% ≈ 150k filas
"""

cost_rand = bq.estimate_cost(query_rand)
print(f"\nRAND() 0.1%: {cost_rand['gb']:.2f} GB (escanea toda la tabla)")

📊 Estimación: 5.440 GB → $0.0332 USD

RAND() 0.1%: 5.44 GB (escanea toda la tabla)


### 5.3 FARM_FINGERPRINT (muestreo reproducible)

Esta es la **estrategia que usamos en nuestro helper**. `FARM_FINGERPRINT` genera un
hash determinista a partir de valores de la fila. El mismo input siempre produce el
mismo hash, lo que garantiza reproducibilidad.

```sql
SELECT *
FROM tabla
WHERE MOD(ABS(FARM_FINGERPRINT(CAST(campo AS STRING))), 1000) < 1
-- Selecciona ~0.1% de forma reproducible
```

**Ventajas:** Reproducible, determinista, distribuido uniformemente.  
**Desventajas:** Escanea toda la tabla (mismo costo). Requiere elegir campos clave.

In [11]:
# Nuestro helper genera queries con FARM_FINGERPRINT automáticamente
sample_sql = bq.sample_query(
    columns="pickup_datetime, trip_distance, fare_amount, tip_amount",
    n=10_000
)
print("Query generada por el helper:")
print(sample_sql)

Query generada por el helper:

            SELECT pickup_datetime, trip_distance, fare_amount, tip_amount
            FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
            ORDER BY FARM_FINGERPRINT(CAST(pickup_datetime AS STRING) || CAST(dropoff_datetime AS STRING))
            LIMIT 10000
            


### 5.4 Comparativa de estrategias

| Estrategia | Costo | Reproducible | Uniforme | Notas |
|------------|-------|:---:|:---:|-------|
| `TABLESAMPLE SYSTEM` | Bajo | No | No* | Muestrea bloques, no filas |
| `RAND()` | Total | No | Sí | Muestreo perfecto pero no repetible |
| `FARM_FINGERPRINT` | Total | Sí | Sí | Nuestra elección para análisis |
| `LIMIT` | Total | No** | No | Solo toma las primeras N filas |

\* TABLESAMPLE puede tener sesgo si los datos están ordenados dentro de bloques.  
\** LIMIT sin ORDER BY es técnicamente no determinista.

**Recomendación:** Usa `TABLESAMPLE` para exploración rápida y `FARM_FINGERPRINT` para
análisis reproducibles. Nuestro helper ya implementa la segunda opción.

---

## 6. Primera Query Real

Vamos a ejecutar nuestra primera query usando el helper. Empezaremos con una
agregación simple para no descargar datos masivos.

In [12]:
# Primero: estimar el costo
query_stats = """
SELECT
    COUNT(*) AS total_viajes,
    ROUND(AVG(trip_distance), 2) AS distancia_promedio_mi,
    ROUND(AVG(fare_amount), 2) AS tarifa_promedio_usd,
    ROUND(AVG(tip_amount), 2) AS propina_promedio_usd,
    ROUND(AVG(total_amount), 2) AS total_promedio_usd,
    ROUND(AVG(passenger_count), 1) AS pasajeros_promedio,
    MIN(pickup_datetime) AS primer_viaje,
    MAX(pickup_datetime) AS ultimo_viaje
FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
"""

bq.estimate_cost(query_stats)

📊 Estimación: 10.881 GB → $0.0664 USD


{'bytes_processed': 11683137600,
 'gb': 10.881,
 'tb': 0.010626,
 'estimated_cost_usd': 0.0664}

In [13]:
# Ejecutar con cache y control de costos
# label: nombre legible para el archivo de cache
# max_cost_usd: límite de seguridad
df_stats = bq.run_query(
    query_stats,
    label="stats_generales_2015",
    max_cost_usd=1.0
)

df_stats

📊 Estimación: 10.881 GB → $0.0664 USD
⏳ Ejecutando query...


/home/danielmf31/Documentos/Documentos_Trabajo/Ingenieria/Programacion/VSCode/Proyectos_personales/Ciencia_Datos/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


✓ Completado: 1 filas en 1.3s
💾 Cache guardado: stats_generales_2015_917120fb64d4.parquet (0.0 MB)


,total_viajes,distancia_promedio_mi,tarifa_promedio_usd,propina_promedio_usd,total_promedio_usd,pasajeros_promedio,primer_viaje,ultimo_viaje
0,146039220,11.85,12.94,1.73,16.1,1.7,2015-01-01 00:00:00+00:00,2015-12-31 23:59:59+00:00


Observa cómo `run_query()` automáticamente:

1. Buscó en el cache local (si ya existía el resultado)
2. Estimó el costo antes de ejecutar
3. Verificó que no excediera `max_cost_usd`
4. Ejecutó la query y mostró tiempo/filas
5. Guardó el resultado en cache como Parquet

Si ejecutas la celda de nuevo, verás **"Cache hit"** en lugar de volver a consultar BigQuery.

In [14]:
# Obtener una muestra de 10k filas para exploración local
query_sample = """
SELECT
    vendor_id,
    pickup_datetime,
    dropoff_datetime,
    passenger_count,
    trip_distance,
    pickup_location_id,
    dropoff_location_id,
    rate_code,
    payment_type,
    fare_amount,
    tip_amount,
    tolls_amount,
    total_amount
FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
WHERE RAND() < 0.0001
LIMIT 10000
"""

df_sample = bq.run_query(
    query_sample,
    label="muestra_10k_exploratoria",
    max_cost_usd=2.0
)

📊 Estimación: 16.671 GB → $0.1018 USD
⏳ Ejecutando query...


/home/danielmf31/Documentos/Documentos_Trabajo/Ingenieria/Programacion/VSCode/Proyectos_personales/Ciencia_Datos/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


✓ Completado: 10,000 filas en 7.9s
💾 Cache guardado: muestra_10k_exploratoria_78cc4eb92f13.parquet (0.3 MB)


---

## 7. Análisis Local con Pandas y NumPy

Una vez que tenemos los datos en un DataFrame (y cacheados en Parquet), podemos
trabajar localmente sin volver a tocar BigQuery.

In [15]:
import numpy as np

# Resumen estadístico
df_sample.describe()

,passenger_count,trip_distance,fare_amount,tip_amount,tolls_amount,total_amount
count,10000.0,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,1.6709,2.974329,12.898400,1.650675,0.290338,15.958123
std,1.328823,3.646758,10.954611,2.275557,1.369240,13.196272
min,1.0,0.000000,-5.000000,-1.440000,0.000000,-6.240000
25%,1.0,1.000000,6.500000,0.000000,0.000000,8.750000
50%,1.0,1.730000,9.500000,1.160000,0.000000,11.800000
75%,2.0,3.160000,14.500000,2.260000,0.000000,17.357500
max,6.0,38.800000,152.000000,27.720000,22.290000,166.310000


In [16]:
# Tipos de datos y memoria
print("=== Tipos de datos ===")
print(df_sample.dtypes)
print(f"\n=== Uso de memoria ===")
print(f"{df_sample.memory_usage(deep=True).sum() / 1024**2:.2f} MB en RAM")

=== Tipos de datos ===
vendor_id                           object
pickup_datetime        datetime64[us, UTC]
dropoff_datetime       datetime64[us, UTC]
passenger_count                      Int64
trip_distance                      float64
pickup_location_id                  object
dropoff_location_id                 object
rate_code                           object
payment_type                        object
fare_amount                        float64
tip_amount                         float64
tolls_amount                       float64
total_amount                       float64
dtype: object

=== Uso de memoria ===
3.04 MB en RAM


In [17]:
# Valores nulos por columna
nulls = df_sample.isnull().sum()
nulls_pct = (nulls / len(df_sample) * 100).round(1)
null_report = pd.DataFrame({'nulos': nulls, 'porcentaje': nulls_pct})
null_report[null_report['nulos'] > 0]

,nulos,porcentaje


In [18]:
# Distribución de métodos de pago
df_sample['payment_type'].value_counts()

payment_type
1    6243
2    3708
3      34
4      15
Name: count, dtype: int64

In [19]:
# Estadísticas rápidas con NumPy
fares = df_sample['fare_amount'].dropna().values

print("=== Tarifas (NumPy) ===")
print(f"Media:    ${np.mean(fares):.2f}")
print(f"Mediana:  ${np.median(fares):.2f}")
print(f"Std:      ${np.std(fares):.2f}")
print(f"P25:      ${np.percentile(fares, 25):.2f}")
print(f"P75:      ${np.percentile(fares, 75):.2f}")
print(f"P99:      ${np.percentile(fares, 99):.2f}")
print(f"Min:      ${np.min(fares):.2f}")
print(f"Max:      ${np.max(fares):.2f}")

=== Tarifas (NumPy) ===
Media:    $12.90
Mediana:  $9.50
Std:      $10.95
P25:      $6.50
P75:      $14.50
P99:      $52.00
Min:      $-5.00
Max:      $152.00


---

## 8. Gestión del Cache

El helper mantiene un cache local en `data/nyc_taxi/cache/`. Veamos qué tenemos.

In [20]:
# Listar archivos en cache
bq.list_cache()

📁 Cache: 2 archivos, 0.3 MB total


,file,size_mb,rows,modified
0,muestra_10k_exploratoria_78cc4eb92f13.parquet,0.26,10000,2026-03-15 18:41:02.414611
1,stats_generales_2015_917120fb64d4.parquet,0.01,1,2026-03-15 18:40:53.996635


---

## 9. Mejores Prácticas

### Lo que aprendimos

1. **Nunca descargues datos masivos crudos.** Siempre agrega en BigQuery y trabaja
   localmente con los resultados agregados o muestras pequeñas.

2. **Selecciona solo las columnas que necesitas.** BigQuery es columnar: `SELECT *`
   escanea todo; seleccionar columnas reduce costos proporcionalmente.

3. **`LIMIT` no reduce costos.** BigQuery debe escanear todos los datos antes de
   aplicar el LIMIT. Usa `TABLESAMPLE` para reducir el escaneo real.

4. **Estima antes de ejecutar.** Siempre usa `estimate_cost()` o `dry_run` para saber
   cuánto vas a gastar.

5. **Cachea todo.** Nuestro helper guarda resultados en Parquet localmente. Si necesitas
   los mismos datos, los lee del disco en lugar de volver a consultar BigQuery.

6. **Usa muestreo reproducible.** `FARM_FINGERPRINT` permite crear muestras
   deterministas para que tus análisis sean replicables.

### Respuesta a nuestra pregunta de negocio

> **¿Cuánto cuesta consultar 1.5 mil millones de viajes?**

Depende de **qué columnas** consultes:

- `SELECT *` sobre toda la tabla: revisa el resultado de la celda de estimación
- Seleccionando solo columnas clave: una fracción del costo total
- Usando agregaciones: el costo de escaneo es igual, pero no descargas datos masivos
- Con el sandbox gratuito (1 TB/mes): puedes hacer docenas de consultas completas al mes

La clave no es evitar las consultas, sino **diseñarlas inteligentemente**.

---

## Siguiente notebook

En **02** exploraremos los datos geoespaciales y temporales para responder:
¿dónde y cuándo se concentran los viajes en NYC?

---

*Notebook creado como parte del NYC Taxi Data Science Project - Fase 1*